In [3]:
import os
os.chdir(os.path.join(os.path.dirname("__file__"), "../.."))

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformer_lens
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import tqdm
from functools import partial

In [7]:
from sae import SparseAutoEncoder

In [8]:
model = transformer_lens.HookedTransformer.from_pretrained("pythia-70m")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-70m into HookedTransformer


In [9]:
EXPANSION = 8
MODEL_PATH = f"saved_models/sae_model_{EXPANSION}x.pt"

# Load saved SAE
d = 512
m = d * EXPANSION

SAE = SparseAutoEncoder(d=d, m=m)
SAE.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
SAE.eval()

device = model.cfg.device
SAE.to(device)

SparseAutoEncoder(
  (act): ReLU()
)

In [10]:
def ablation_hook(resid, hook, feature_idx):
    f = SAE.encoder(resid)
    contr = f[..., feature_idx].unsqueeze(-1) * SAE.W_dec.data[feature_idx]
    return resid - contr

In [11]:
text = "In a new report from Reuters, the governor of New York "
# text = "The governor of New York announced legislation to improve public transportation in the state."
# text = "The governor of New"

In [12]:
IDX = 3440

ablated_logits = model.run_with_hooks(
    text,
    fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=IDX))]
)

In [13]:
clean_logits = model(text)

In [14]:
clean_logprobs = F.log_softmax(clean_logits.float(), dim=-1)
ablated_logprobs = F.log_softmax(ablated_logits.float(), dim=-1)
clean_probs = clean_logprobs.exp()

kl_div = (clean_probs * (clean_logprobs - ablated_logprobs)).sum(dim=-1)
print(f"KL divergence per position: {kl_div.squeeze()}")
print(f"Mean KL divergence: {kl_div.mean().item():.6f}")

clean_top = clean_logits.argmax(dim=-1).squeeze()
ablated_top = ablated_logits.argmax(dim=-1).squeeze()

tokenizer = model.tokenizer
for pos in range(len(clean_top)):
    if clean_top[pos] != ablated_top[pos]:
        print(f"Position {pos}: {tokenizer.decode(clean_top[pos])} → {tokenizer.decode(ablated_top[pos])}")

KL divergence per position: tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        1.3324e-02, 2.6020e-03, 9.1230e-04, 2.1212e-05, 1.5196e-02, 6.2669e-04,
        2.2904e-04, 1.3820e-06], device='mps:0', grad_fn=<SqueezeBackward0>)
Mean KL divergence: 0.002351


In [15]:
tokens = model.to_str_tokens(text)

# KL per token position
print("KL divergence per token:\n")
for pos, (tok, kl) in enumerate(zip(tokens, kl_div.squeeze())):
    print(f"{pos:2d} {tok:>15s}  KL={kl:.6f}")

# Top-5 predictions comparison
k = 5
print("\nTop-5 predictions (clean vs ablated):\n")
for pos in range(len(tokens)):
    clean_topk = clean_logits[0, pos].topk(k)
    ablated_topk = ablated_logits[0, pos].topk(k)
    print(f"Position {pos}: '{tokens[pos]}'")
    print(f"  Clean:   {[tokenizer.decode(t) for t in clean_topk.indices]}")
    print(f"  Ablated: {[tokenizer.decode(t) for t in ablated_topk.indices]}")
    print()

KL divergence per token:

 0   <|endoftext|>  KL=0.000000
 1              In  KL=0.000000
 2               a  KL=0.000000
 3             new  KL=0.000000
 4          report  KL=0.000000
 5            from  KL=0.000000
 6         Reuters  KL=0.013324
 7               ,  KL=0.002602
 8             the  KL=0.000912
 9        governor  KL=0.000021
10              of  KL=0.015196
11             New  KL=0.000627
12            York  KL=0.000229
13                  KL=0.000001

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['Q', 'The', '\n', 'A', '1']

Position 1: 'In'
  Clean:   [' the', ' a', 'fluence', ' this', ' an']
  Ablated: [' the', ' a', 'fluence', ' this', ' an']

Position 2: ' a'
  Clean:   [' new', ' recent', ' conventional', ' news', ' typical']
  Ablated: [' new', ' recent', ' conventional', ' news', ' typical']

Position 3: ' new'
  Clean:   [' study', ' paper', ' report', ' book', ' version']
  Ablated: [' 

In [39]:
# Greedy generation: clean vs ablated
def generate_with_ablation(text, feature_idx, max_new_tokens=5):
    clean_tokens = [text]
    ablated_tokens = [text]

    clean_input = text
    ablated_input = text

    for _ in range(max_new_tokens):
        clean_logits = model(clean_input)
        next_clean = tokenizer.decode(clean_logits[0, -1].argmax())
        clean_input += next_clean
        clean_tokens.append(next_clean)

        ablated_logits = model.run_with_hooks(
            ablated_input,
            fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=feature_idx))]
        )
        next_ablated = tokenizer.decode(ablated_logits[0, -1].argmax())
        ablated_input += next_ablated
        ablated_tokens.append(next_ablated)

    print(f"Clean:   {clean_input}")
    print(f"Ablated: {ablated_input}")
    # print(clean_tokens)
    # print(ablated_tokens)

generate_with_ablation(text, IDX)

Clean:   The governor of New York City is a former
Ablated: The governor of New South Wales, Andrew Jackson
